In [ ]:
import pandas as pd
from utils import merge_csv_files_in_directory, standardize_labels
from IPython.display import display, HTML

df = pd.read_csv("merged.csv")
# df = merge_csv_files_in_directory()

#remove duplicates
df = df.drop_duplicates(keep='first')

df.drop("pr_title",inplace = True,axis=1)
df.drop("class_names",inplace = True,axis=1)
df.drop("commit_sha",inplace = True,axis=1)
df.drop("commit_message",inplace = True,axis=1)
df.drop("file_path",inplace = True,axis=1)
df.drop("additions",inplace = True,axis=1)
df.drop("deletions",inplace = True,axis=1)
df.drop("file_name",inplace = True,axis=1)
# df.drop("full_repo_name",inplace = True,axis=1)

df.drop("content_before",inplace = True,axis=1)
df.drop("content_after",inplace = True,axis=1)

# df.drop("patch",inplace = True,axis=1)

df.drop("created_at",inplace = True,axis=1)
df.drop("closed_at",inplace = True,axis=1)
df.drop("status",inplace = True,axis=1)

# Standardize labels
labels = ['bug', 'enhancement', 'performance', 'refactor', 'clean']
df = standardize_labels(df, labels)

### Limit df size for testing purposes ###
label_enh = df[df['labels'] == 'enhancement']
label_bug = df[df['labels'] == 'bug']
label_cln = df[df['labels'] == 'clean']
# label_prf = df[df['labels'] == 'performance']
# label_ref = df[df['labels'] == 'refactor']
sampled_enh = label_enh.sample(n=15300, random_state=1)
sampled_bug = label_bug.sample(n=15300, random_state=1)
sampled_cln = label_cln.sample(n=15300, random_state=1)
# sampled_prf = label_prf.sample(n=3200, random_state=1)
# sampled_ref = label_ref.sample(n=3200, random_state=1)
df = pd.concat([sampled_enh, sampled_bug, sampled_cln])

resulting_label_count = df['labels'].nunique()

# # Filter NaN rows
# df = df.dropna(subset=['content_before', 'content_after'])
df = df.dropna(subset=['patch'])

# Display DataFrame with a scrollable table
#display(HTML(df.to_html(max_rows=None, max_cols=None)))

# Count the occurrences of each unique value in the 'lables' column
label_counts = df['labels'].value_counts()
print(label_counts)

df

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "microsoft/graphcodebert-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to("cuda:0")

In [ ]:
import os

# Directory to save embeddings
save_dir = "embeddings/pooler_GCB_embeddings_patch_before_largewithcodedata_15300limit3label"
os.makedirs(save_dir, exist_ok=True)

In [ ]:
from utils import extract_code_from_patch

# Tokenizing using subtracting full code before and after embeddings

# def generate_and_save_embeddings_batch(contents_before, contents_after, indices):
def generate_and_save_embeddings_batch(patches, indices):
    patches_before = []

    for patch in patches:
        patch_before, patch_after = extract_code_from_patch(patch)
        patches_before.append(patch_before)
    
    
    # Tokenize the before and after code patch snippet
    tokens_before = tokenizer(patches_before, truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")

    # Tokenize the before and after code snippets
    # tokens_before = tokenizer(list(contents_before), truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")
    # tokens_after = tokenizer(list(contents_after), truncation=True, padding='max_length', max_length=512, return_tensors='pt').to("cuda:0")

    # Get the embeddings for both tokenized sequences
    with torch.no_grad():

        embeddings_before = model(**tokens_before).pooler_output #pooler
        # embeddings_after = model(**tokens_after).pooler_output
    
    # Subtract the embeddings
    # subtracted_embeddings = embeddings_after - embeddings_before
    # print(subtracted_embeddings.shape)
    
    # Average the subtracted embeddings across the token dimension
    # subtracted_embeddings = subtracted_embeddings.mean(dim=1)
    
    save_path = os.path.join(save_dir, f"embedding_{indices[0].item()}.pt")
    torch.save(embeddings_before, save_path)
    

In [ ]:
from torch.utils.data import DataLoader, Dataset

# Custom Dataset Class to handle the dataframe
class CodeDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        # content_before = row['content_before']
        # content_after = row['content_after']
        # return content_before, content_after, idx
        
        patch = row['patch']
        return patch, idx

In [ ]:
import tqdm

# Creating a DataLoader
def create_dataloader(dataframe, batch_size=32, shuffle=False):
    dataset = CodeDataset(dataframe)
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)

batch_size = 128
dataloader = create_dataloader(df, batch_size=batch_size)

for batch in tqdm.notebook.tqdm(dataloader):
    
    # contents_before, contents_after, indices = batch
    # generate_and_save_embeddings_batch(contents_before, contents_after, indices)
    
    patch, indices = batch
    generate_and_save_embeddings_batch(patch, indices)
    


In [ ]:
## Alternate starting node, if embeddings will be loaded
import pandas as pd
from utils import merge_csv_files_in_directory, standardize_labels
from IPython.display import display, HTML
from utils import load_embeddings_from_directory, merge_csv_files_in_directory, standardize_labels

df = pd.read_csv("merged.csv")
# df = merge_csv_files_in_directory()

#remove duplicates
df = df.drop_duplicates(keep='first')

df.drop("pr_title",inplace = True,axis=1)
df.drop("class_names",inplace = True,axis=1)
df.drop("commit_sha",inplace = True,axis=1)
df.drop("commit_message",inplace = True,axis=1)
df.drop("file_path",inplace = True,axis=1)
df.drop("additions",inplace = True,axis=1)
df.drop("deletions",inplace = True,axis=1)
df.drop("file_name",inplace = True,axis=1)
# df.drop("full_repo_name",inplace = True,axis=1)

df.drop("content_before",inplace = True,axis=1)
df.drop("content_after",inplace = True,axis=1)

# df.drop("patch",inplace = True,axis=1)

df.drop("created_at",inplace = True,axis=1)
df.drop("closed_at",inplace = True,axis=1)
df.drop("status",inplace = True,axis=1)

# Standardize labels
labels = ['bug', 'enhancement', 'performance', 'refactor', 'clean']
df = standardize_labels(df, labels)
resulting_label_count = df['labels'].nunique()

# Count the occurrences of each unique value in the 'lables' column
label_counts = df['labels'].value_counts()
print(label_counts)

### Limit df size for testing purposes ###
label_enh = df[df['labels'] == 'enhancement']
label_bug = df[df['labels'] == 'bug']
label_cln = df[df['labels'] == 'clean']
# label_prf = df[df['labels'] == 'performance']
# label_ref = df[df['labels'] == 'refactor']
sampled_enh = label_enh.sample(n=15300, random_state=1)
sampled_bug = label_bug.sample(n=15300, random_state=1)
sampled_cln = label_cln.sample(n=15300, random_state=1)
# sampled_prf = label_prf.sample(n=3200, random_state=1)
# sampled_ref = label_ref.sample(n=3200, random_state=1)
df = pd.concat([sampled_enh, sampled_bug, sampled_cln])

# Refreshing label count
resulting_label_count = df['labels'].nunique()

# # Filter NaN rows
# df = df.dropna(subset=['content_before', 'content_after'])
df = df.dropna(subset=['patch'])

# Display DataFrame with a scrollable table
#display(HTML(df.to_html(max_rows=None, max_cols=None)))

# Count the occurrences of each unique value in the 'lables' column
label_counts = df['labels'].value_counts()
print(label_counts)

df

In [1]:
import torch
from utils import load_embeddings_from_directory

# # Load embeddings from embeddings folder
all_embeddings = load_embeddings_from_directory('embeddings/GCB_LHS_embeddings_patch_before_largewithcodedata_15300limit3label')

# Convert the embeddings to a DataFrame or add them to the original DataFrame
#df['tokens'] = list(torch.chunk(torch.vstack(all_embeddings).cpu(), df.shape[0], dim=0))
df

  0%|          | 0/359 [00:00<?, ?it/s]

embedding_0.pt
torch.Size([128, 768])
embedding_128.pt
torch.Size([128, 768])
embedding_256.pt
torch.Size([128, 768])
embedding_384.pt
torch.Size([128, 768])
embedding_512.pt
torch.Size([128, 768])
embedding_640.pt
torch.Size([128, 768])
embedding_768.pt
torch.Size([128, 768])
embedding_896.pt
torch.Size([128, 768])
embedding_1024.pt
torch.Size([128, 768])
embedding_1152.pt
torch.Size([128, 768])
embedding_1280.pt
torch.Size([128, 768])
embedding_1408.pt
torch.Size([128, 768])
embedding_1536.pt
torch.Size([128, 768])
embedding_1664.pt
torch.Size([128, 768])
embedding_1792.pt
torch.Size([128, 768])
embedding_1920.pt
torch.Size([128, 768])
embedding_2048.pt
torch.Size([128, 768])
embedding_2176.pt
torch.Size([128, 768])
embedding_2304.pt
torch.Size([128, 768])
embedding_2432.pt
torch.Size([128, 768])
embedding_2560.pt
torch.Size([128, 768])
embedding_2688.pt
torch.Size([128, 768])
embedding_2816.pt
torch.Size([128, 768])
embedding_2944.pt
torch.Size([128, 768])
embedding_3072.pt
torch.Si

NameError: name 'df' is not defined

In [ ]:
# # Tokenizing using patch
# import torch
# from utils import extract_code_from_patch

# def tokenize_code(row):

#     patch_before, patch_after = extract_code_from_patch(row['patch'])
    
#     # Tokenize the before and after code snippets
#     tokens_before = tokenizer(patch_before, truncation=True, padding='max_length', max_length=512, return_tensors='pt')
#     tokens_after = tokenizer(patch_after, truncation=True, padding='max_length', max_length=512, return_tensors='pt')
    
#     # Get the embeddings for both tokenized sequences
#     with torch.no_grad():
#         embeddings_before = model(**tokens_before).last_hidden_state
#         embeddings_after = model(**tokens_after).last_hidden_state
    
#     # Ensure the two tensors are of the same shape before subtracting
#     assert embeddings_before.shape == embeddings_after.shape, "Shape mismatch between before and after embeddings"
    
#     # Subtract the embeddings
#     subtracted_embeddings = embeddings_after - embeddings_before
    
#     # Average the subtracted embeddings across the token dimension
#     # to get a single vector representation for each input. (2D tensor instead of 3D)
#     subtracted_embeddings = subtracted_embeddings.mean(dim=1)
    
#     return subtracted_embeddings

# df['tokens'] = df.apply(tokenize_code, axis=1)
df

In [ ]:
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression # Classification
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
import pandas as pd
import torch

# One-hot encoding ile labels sütununu dönüştürme
df_encoded_labels = pd.get_dummies(df, columns=['labels'])
df_encoded_repo = pd.get_dummies(df, columns=['full_repo_name'])

# Özellikler ve etiketler
X = torch.stack(tuple(df['tokens'].values))
X_repo = df_encoded_repo.drop(columns=['labels','patch','tokens'])
X_repo = X_repo.astype(int)

# y = df_encoded.drop(columns=['content_before', 'content_after', 'tokens'])
y = df_encoded_labels.drop(columns=['patch', 'tokens', 'full_repo_name'])
y = y.astype(int) # convert it 0,1     instead of true, false 

# Split the code data into test, train and validation sets
X_train, X_test, X_repo_train, X_repo_test, y_train, y_test = train_test_split(X, X_repo, y, test_size=0.3, random_state=42) #random_state: 0,3074,42
X_test, X_val, X_repo_test, X_repo_val, y_test, y_val = train_test_split(X_test, X_repo_test, y_test, test_size=0.5, random_state=42)



In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Flattening the 3D tensor to 2D
X_train = X_train.view(X_train.size(0), -1) #%70
X_test = X_test.view(X_test.size(0), -1)  #%15
X_val = X_val.view(X_val.size(0), -1)   #%15

# # Scaling X values so that the mean is 0 and the st.dev. is 1
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)

# # Converting inputs back to tensors
# X_train = torch.tensor(X_train_scaled, dtype=torch.float32)
# X_test = torch.tensor(X_test_scaled, dtype=torch.float32)

# Turning X_repo into tensors
X_repo_train = torch.tensor(X_repo_train.to_numpy(), dtype=torch.float32)
X_repo_test = torch.tensor(X_repo_test.to_numpy(), dtype=torch.float32)
X_repo_val = torch.tensor(X_repo_val.to_numpy(), dtype=torch.float32)

# Turning y into tensors
y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
y_test = torch.tensor(y_test.to_numpy(), dtype=torch.float32)
y_val = torch.tensor(y_val.to_numpy(), dtype=torch.float32)

print(y_train.shape)
print(X_train.shape)
print(X_repo_train.shape)

In [ ]:
# Create DataLoader
batch_size = 128
train_dataset = TensorDataset(X_train, X_repo_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
import torch.nn as nn

# Define the ANN model
class CodeDiffNN(nn.Module):
    def __init__(self, code_input_size, repo_input_size, hidden_sizes, output_size, dropout_rate=0.001):
        super(CodeDiffNN, self).__init__()
        
        # Pipeline for code embeddings
        self.code_fc1 = nn.Linear(code_input_size, hidden_sizes[0])
        self.code_relu1 = nn.ReLU()
        self.code_dropout1 = nn.Dropout(dropout_rate)
        
        self.code_fc2 = nn.Linear(hidden_sizes[0], hidden_sizes[1])
        self.code_relu2 = nn.ReLU()
        self.code_dropout2 = nn.Dropout(dropout_rate)
        
        # Pipeline for repo names
        self.repo_fc1 = nn.Linear(repo_input_size, hidden_sizes[0])
        self.repo_relu1 = nn.ReLU()
        self.repo_dropout1 = nn.Dropout(dropout_rate)
        
        self.repo_fc2 = nn.Linear(hidden_sizes[0], hidden_sizes[1])
        self.repo_relu2 = nn.ReLU()
        self.repo_dropout2 = nn.Dropout(dropout_rate)
        
        # Combining the outputs of both pipelines
        combined_input_size = hidden_sizes[1] * 2  # Since we're combining two pipelines
        self.fc_combined = nn.Linear(combined_input_size, hidden_sizes[2])
        self.relu_combined = nn.ReLU()
        self.dropout_combined = nn.Dropout(dropout_rate)

        # Output layer
        self.fc_out = nn.Linear(hidden_sizes[2], output_size)
        
    def forward(self, code_x, repo_x):
        # Process code embeddings
        code_x = self.code_fc1(code_x)
        code_x = self.code_relu1(code_x)
        code_x = self.code_dropout1(code_x)
        
        code_x = self.code_fc2(code_x)
        code_x = self.code_relu2(code_x)
        code_x = self.code_dropout2(code_x)
        
        # Process repo names
        repo_x = self.repo_fc1(repo_x)
        repo_x = self.repo_relu1(repo_x)
        repo_x = self.repo_dropout1(repo_x)
        
        repo_x = self.repo_fc2(repo_x)
        repo_x = self.repo_relu2(repo_x)
        repo_x = self.repo_dropout2(repo_x)
        
        # Combine both outputs
        combined_x = torch.cat((code_x, repo_x), dim=1)
        
        # Process the combined output
        combined_x = self.fc_combined(combined_x)
        combined_x = self.relu_combined(combined_x)
        combined_x = self.dropout_combined(combined_x)
        
        # Final output
        output = self.fc_out(combined_x)
        return output


In [ ]:
# Define model parameters
print(resulting_label_count)

code_input_size = X_train.shape[1]
repo_input_size = X_repo_train.shape[1]
hidden_sizes = [512, 256, 256]
output_size = resulting_label_count

device = "cuda:0"
model = CodeDiffNN(code_input_size, repo_input_size, hidden_sizes, output_size).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-5)

In [ ]:
model

In [ ]:
import copy

# Function to calculate accuracy
def calculate_accuracy(outputs, labels):
    _, predicted = torch.max(outputs, 1)  # Get the index of the max log-probability
    labels_1d = torch.argmax(labels, dim=1) # turn into 1D tensor
    correct = (predicted == labels_1d).sum().item()
    accuracy = correct / labels.size(0)  # Accuracy = number of correct predictions / total number of samples
    return accuracy


# Train the model

# Initialize variables to track the best model and best accuracy
best_accuracy = 0.0
best_model = None

# Send validation data to GPU
y_val = y_val.to(device)
X_val = X_val.to(device)
X_repo_val = X_repo_val.to(device)

num_epochs = 500
for epoch in range(num_epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0
    running_accuracy = 0.0
    
    for batch_X, batch_X_repo, batch_y in train_loader:
        # Zero the parameter gradients
        optimizer.zero_grad()
    
        # Forward pass
        batch_y = batch_y.to(device)
        batch_X = batch_X.to(device)
        batch_X_repo = batch_X_repo.to(device)
        
        outputs = model(batch_X, batch_X_repo)
        loss = criterion(outputs, batch_y)
    
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Calculate accuracy
        accuracy = calculate_accuracy(outputs, batch_y)
        
        # Update running loss and accuracy
        running_loss += loss.item()
        running_accuracy += accuracy
    
    # Print average loss and accuracy for the epoch
    avg_loss = running_loss / len(train_loader)
    avg_accuracy = running_accuracy / len(train_loader)
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}, Accuracy: {avg_accuracy:.4f}')

    # Evaluate on validation data
    model.eval()  # Set the model to evaluation mode
    with torch.no_grad():  # Disable gradient computation for validation
        val_outputs = model(X_val, X_repo_val)
        val_loss = criterion(val_outputs, y_val)
        val_accuracy = calculate_accuracy(val_outputs, y_val)
    
    print(f'Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}')
    
    # Check if the current model has the best validation accuracy
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_model = copy.deepcopy(model.state_dict())  # Save the model with the best accuracy


In [ ]:
with torch.no_grad():
    # Load best model from previous training cycle
    model.load_state_dict(best_model)
    print(f'Best model accuracy on validation data: {best_accuracy:.4f}')

    model.eval()
    test_outputs = model(X_test.to("cuda:0"), X_repo_test.to("cuda:0"))
    
    accuracy = calculate_accuracy(test_outputs, y_test.to("cuda:0"))
    print(f'Accuracy on test data: {accuracy:.4f}')

In [ ]:
temp_df = pd.read_csv("merged.csv")

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

_, predicted = torch.max(test_outputs.detach().cpu(), 1)  # Get the index of the max log-probability
labels_1d = torch.argmax(y_test.detach().cpu(), dim=1) # turn into 1D tensor

# Convert to numpy arrays
y_test_np = labels_1d.numpy()
predicted_np = predicted.numpy()

# Compute confusion matrix
cm = confusion_matrix(y_test_np, predicted_np)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title('Confusion Matrix for Multi-Class Classification')
plt.show()

In [ ]:
outputs.shape

In [ ]:
### DEPRECATED ###

In [ ]:

# # Örnek tahminler
# sample_data = pd.DataFrame({
#     'file_name': ['ProtoSubject.java','BasicAnnotationProcessor.java'], #1:enhancement, 2:enhancement
#     'full_repo_name': ['google/truth','google/auto']
# })

# # Kodlama işlemleri
# sample_data['file_name_encoded'] = le_class.transform(sample_data['file_name'])
# sample_data['full_repo_name_encoded'] = le_repo.transform(sample_data['full_repo_name'])

# # Özellikleri hazırlama
# X_sample = sample_data[['file_name_encoded', 'full_repo_name_encoded']]
# X_sample = torch.tensor(X_sample.to_numpy(), dtype=torch.float32)

# with torch.no_grad():
#     model.eval()
#     test_outputs = model(X_sample)
#     predicted = torch.sigmoid(test_outputs)  # Sigmoid fonksiyonu uygulanmalı
#     print( predicted)
#     predicted = (predicted > 0.5).float()  # 0.5'ten büyük olanlar 1, diğerleri 0
#     print("after ceiling")
#     print( predicted)